# PromptForge — Démo complète

> Optimisation automatique de prompts par boucle agentique (Génère → Évalue → Itère)

Ce notebook démontre PromptForge sur plusieurs datasets. Il couvre :
- le lancement d'un run avec visualisation de la progression
- le suivi des coûts API en temps réel
- la comparaison prompt initial / prompt optimisé
- les feedbacks détaillés du juge

## 0. Setup

In [1]:
%pip install -q anthropic tenacity rich python-dotenv plotly pandas nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
sys.path.insert(0, '..')

# Charge ANTHROPIC_API_KEY depuis ../.env si présent
from dotenv import load_dotenv
load_dotenv('../.env')

# Jupyter tourne dans une boucle asyncio — nest_asyncio permet d'utiliser asyncio.run() dedans
import nest_asyncio
nest_asyncio.apply()

from core.loop import PromptForge
from datasets.examples import DATASETS
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

print(f"Datasets disponibles : {list(DATASETS.keys())}")
print(f"Clé API configurée   : {'✓' if os.getenv('ANTHROPIC_API_KEY') else '✗ manquante'}")

Datasets disponibles : ['summarization', 'sentiment', 'extraction', 'code', 'address', 'language', 'ticket', 'regex', 'feedback', 'security']
Clé API configurée   : ✓


## 1. Lancement de l'optimisation

In [14]:
# ── Choix du dataset ────────────────────────────────────────────────────────
# Datasets simples   : summarization, sentiment, extraction, code, address, language, ticket
# Datasets moyens    : regex, feedback
# Dataset complexe   : security

DATASET_NAME = 'code'   # ← modifier ici pour changer de dataset

dataset = DATASETS[DATASET_NAME]

print(f"Dataset    : {DATASET_NAME}")
print(f"Tâche      : {dataset['task']}")
print(f"Prompt initial : {dataset['initial_prompt']}")
print(f"Exemples   : {len(dataset['examples'])}")

print(f"================\nExemple concret\n================ \nInput : {dataset['examples'][0]['input']}\n=> Résultat attendu :\n {dataset['examples'][0]['expected_output']}")

Dataset    : code
Tâche      : Générer une fonction Python propre et documentée à partir d'une description
Prompt initial : Écris du code Python pour faire ça :
Exemples   : 2
Exemple concret
Input : Une fonction qui calcule la moyenne d'une liste de nombres en ignorant les valeurs None
=> Résultat attendu :
 def calculate_mean(values: list) -> float:
    """Calcule la moyenne en ignorant les None."""
    clean = [v for v in values if v is not None]
    return sum(clean) / len(clean) if clean else 0.0


In [15]:
import importlib, core.loop
importlib.reload(core.loop)
from core.loop import PromptForge

forge = PromptForge(
    model='claude-haiku-4-5-20251001',
    n_variants=4,
    n_iterations=3,
    top_k=2,
    execute=False,   # True = exécution réelle (~3× plus d'appels API, plus fiable)
)

# await évite asyncio.run() et le conflit de contexte Python 3.12 avec nest_asyncio
run = await forge.arun(
    initial_prompt=dataset['initial_prompt'],
    examples=dataset['examples'],
    task_description=dataset['task'],
    verbose=True,
)


🔄 Itération 1/3
📝 5 candidats (dont 1 parents conservés)
  ✓ Score : 0.50/10
  ✓ Score : 6.50/10
  ✓ Score : 7.20/10
  ✓ Score : 6.50/10
  ✓ Score : 7.50/10

🏆 Meilleur score itération 1: 7.50/10
💬 Feedback: Le prompt est clair et demande du code de qualité production avec bonnes pratiques (PEP 8, docstrings, gestion erreurs). Points forts : spécification ...

🔄 Itération 2/3
📝 6 candidats (dont 2 parents conservés)
  ✓ Score : 8.20/10
  ✓ Score : 8.50/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10

🏆 Meilleur score itération 2: 8.50/10
💬 Feedback: Le prompt est très clair et structuré avec des critères précis (type hints, docstring détaillée, gestion d'erreurs, exemples). Il définit bien les att...

🔄 Itération 3/3
📝 6 candidats (dont 2 parents conservés)
  ✓ Score : 8.50/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10
  ✓ Score : 8.20/10

🏆 Meilleur score itération 3: 8.50/10
💬 Feedback: Prompt très bien structur

    Coûts API — session     
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Métrique      ┃   Valeur ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ Appels API    │       24 │
│ Tokens entrée │   13,987 │
│ Tokens sortie │    6,481 │
│ Cache read    │ 0 (0.0%) │
│ Cache write   │        0 │
│ Coût estimé   │ ~$0.0464 │
└───────────────┴──────────┘


🏆 Prompt optimisé :
Je dois implémenter cette fonctionnalité Python : [DESCRIPTION]. Fournis le code source complet avec : 1) type hints explicites pour paramètres et retour, 2) une docstring détaillée (description brève, paramètres, valeur retournée, exceptions levées), 3) la gestion des cas limites spécifiques (None, listes vides, valeurs invalides) avec ValueError ou TypeError appropriés, 4) deux exemples d'utilisation en commentaire illustrant un cas normal et un cas d'erreur. Le code doit être production-ready, concis et suivre PEP 8 sans sur-ingénierie.


## 2. Visualisation de la progression

In [16]:
iterations  = [it.iteration  for it in run.iterations]
best_scores = [it.best_score for it in run.iterations]
avg_scores  = [it.avg_score  for it in run.iterations]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=iterations, y=best_scores,
    mode='lines+markers',
    name='Meilleur score',
    line=dict(color='#2ecc71', width=3),
    marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=iterations, y=avg_scores,
    mode='lines+markers',
    name='Score moyen',
    line=dict(color='#3498db', width=2, dash='dash'),
    marker=dict(size=8),
))
fig.add_hline(
    y=run.iterations[0].avg_score,
    line_dash='dot', line_color='grey',
    annotation_text='Score initial',
    annotation_position='bottom right',
)

fig.update_layout(
    title='📈 Progression des scores par itération',
    xaxis_title='Itération',
    yaxis_title='Score (0-10)',
    yaxis=dict(range=[0, 10]),
    template='plotly_white',
    legend=dict(x=0.02, y=0.98),
)
fig.show()

## 3. Distribution des scores — dernière itération

In [17]:
last_iteration = run.iterations[-1]
df = pd.DataFrame(last_iteration.candidates)
df = df.sort_values('score', ascending=False).reset_index(drop=True)

fig2 = px.bar(
    df,
    x=df.index,
    y='score',
    color='score',
    color_continuous_scale='RdYlGn',
    range_color=[0, 10],
    title=f'📊 Scores des candidats — Itération {last_iteration.iteration}',
    labels={'x': 'Candidat (trié par score)', 'score': 'Score /10'}
)
fig2.update_layout(template='plotly_white', showlegend=False)
fig2.show()

## 4. Comparaison Prompt Initial vs Prompt Optimisé

In [18]:
print('=' * 60)
print('PROMPT INITIAL')
print('=' * 60)
print(run.initial_prompt)

print()
print('=' * 60)
print(f'PROMPT OPTIMISÉ  (score : {run.best_score:.2f}/10)')
print('=' * 60)
print(run.best_prompt)

improvement = run.best_score - run.iterations[0].avg_score
print(f'\n📈 Amélioration : +{improvement:.2f} points vs score moyen initial')

PROMPT INITIAL
Écris du code Python pour faire ça :

PROMPT OPTIMISÉ  (score : 8.50/10)
Je dois implémenter cette fonctionnalité Python : [DESCRIPTION]. Fournis le code source complet avec : 1) type hints explicites pour paramètres et retour, 2) une docstring détaillée (description brève, paramètres, valeur retournée, exceptions levées), 3) la gestion des cas limites spécifiques (None, listes vides, valeurs invalides) avec ValueError ou TypeError appropriés, 4) deux exemples d'utilisation en commentaire illustrant un cas normal et un cas d'erreur. Le code doit être production-ready, concis et suivre PEP 8 sans sur-ingénierie.

📈 Amélioration : +2.86 points vs score moyen initial


## 5. Détail des feedbacks du judge

In [8]:
last_iteration = run.iterations[-1]
df_last = pd.DataFrame(last_iteration.candidates).sort_values('score', ascending=False).reset_index(drop=True)

print(f'Top 3 prompts — Itération {last_iteration.iteration} :\n')
for i, row in df_last.head(3).iterrows():
    print(f'--- Candidat #{i+1} | Score : {row["score"]:.2f}/10 ---')
    print(f'Prompt   : {row["prompt"][:150]}...' if len(row["prompt"]) > 150 else f'Prompt   : {row["prompt"]}')
    print(f'Feedback : {row["feedback"]}')
    print()

Top 3 prompts — Itération 5 :

--- Candidat #1 | Score : 8.50/10 ---
Prompt   : Tu es un expert Python senior spécialisé en code production. Tâche : [DESCRIPTION]

Génère une fonction suivant cette structure immuable :

**1. Signa...
Feedback : Prompt très structuré et précis avec des instructions claires et immuables. Points forts : spécification détaillée de chaque section (signature, validation, docstring Google), contraintes PEP 8, limite de 30 lignes. Points d'amélioration : la contrainte de concision (30 lignes) peut entrer en conflit avec la validation exhaustive demandée; absence d'exemple concret du prompt lui-même rendant difficile la vérification de la sortie réelle; critère 'cas limites identifiables' reste subjectif selon le contexte de [DESCRIPTION].

--- Candidat #2 | Score : 8.50/10 ---
Prompt   : Tu es un expert Python chargé de générer du code production. Tâche : [DESCRIPTION]

**Règles non négociables :**

1. **Type hints systématiques** : To...
Feedback : Prompt trè

In [9]:
import json
from pathlib import Path

output = {
    "dataset": DATASET_NAME,
    "task": dataset['task'],
    "initial_prompt": run.initial_prompt,
    "optimized_prompt": run.best_prompt,
    "best_score": run.best_score,
    "iterations": len(run.iterations),
    "cost_summary": run.cost_summary or {},
}

out_path = Path(f'../results/best_prompt_{DATASET_NAME}.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2))

print(f"✅ Prompt optimisé exporté : {out_path}")
print(f"\n{run.best_prompt}")

✅ Prompt optimisé exporté : ../results/best_prompt_code.json

Tu es un expert en développement Python. Écris une fonction Python pour : [DESCRIPTION]

Exigences de qualité :
- **Docstring Google** : description, Args (type + description), Returns (type + description), Raises (exceptions avec conditions)
- **Type hints** explicites pour tous les paramètres et la valeur retournée
- **Gestion d'erreurs précise** : TypeError pour argument mal typé, ValueError pour valeur invalide, autres selon contexte. Messages d'erreur clairs
- **Code PEP 8** : nommage, indentation, longueur de lignes
- **Exemple d'utilisation** : 1-2 lignes commentées montrant entrée/sortie (ex: # fonction(5) → 25)
- **Cas limites** : considérer valeurs nulles, listes vides, nombres négatifs, strings vides selon contexte
- **Concision** : pas de sur-ingénierie, max 30 lignes docstring incluse, pas de dépendances externes

Adapte la complexité exactement à la tâche décrite.

Fournir la fonction complète, prête pour la pr

## 7. Export du prompt optimisé

In [10]:
# Résumé des coûts depuis le run (stocké dans run.cost_summary si CostTracker intégré)
cost = run.cost_summary or {}

if cost:
    labels = ['Tokens entrée', 'Tokens sortie', 'Cache read', 'Cache write']
    values = [
        cost.get('input_tokens', 0),
        cost.get('output_tokens', 0),
        cost.get('cache_read_tokens', 0),
        cost.get('cache_write_tokens', 0),
    ]

    fig_cost = go.Figure(go.Pie(
        labels=labels,
        values=values,
        hole=0.4,
        marker_colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'],
    ))
    fig_cost.update_layout(
        title=f"💰 Répartition des tokens — coût estimé : ~${cost.get('total_cost_usd', 0):.4f}",
        template='plotly_white',
    )
    fig_cost.show()

    print(f"Appels API      : {cost.get('api_calls', 'N/A')}")
    print(f"Tokens entrée   : {cost.get('input_tokens', 0):,}")
    print(f"Tokens sortie   : {cost.get('output_tokens', 0):,}")
    print(f"Cache read      : {cost.get('cache_read_tokens', 0):,}  ({cost.get('cache_hit_pct', 0)}% hit rate)")
    print(f"Coût estimé     : ~${cost.get('total_cost_usd', 0):.4f}")
else:
    print("ℹ️  Données de coût non disponibles (CostTracker pas encore intégré au run).")

Appels API      : 42
Tokens entrée   : 28,955
Tokens sortie   : 13,994
Cache read      : 0  (0.0% hit rate)
Coût estimé     : ~$0.0989


## 6. Coûts API